# MovieLens Analytics with Spark DataFrames

This notebook reads MovieLens CSV files from HDFS, applies explicit schemas, performs aggregations and a join, then writes the result back to the student's HDFS directory.

## Learning objectives

- Read CSV data from HDFS with an explicit schema.
- Validate row counts and key fields.
- Aggregate ratings and join the result with movie details.
- Explain the difference between many output part files and `coalesce(1)`.

## 1. Start HDFS and Spark

Run these commands in a WSL terminal before starting the notebook:

```bash
start-dfs.sh
/opt/spark/sbin/start-master.sh
/opt/spark/sbin/start-worker.sh "spark://$(hostname):7077"
jps
hdfs dfsadmin -report
```

Confirm that `jps` lists the NameNode, DataNode, SecondaryNameNode, Master, and Worker. The HDFS report must show one live DataNode, and [http://localhost:8080](http://localhost:8080) must show one live Spark worker.

## 2. Prepare the MovieLens data in HDFS

The Windows files are expected at `C:\data\movies.csv` and `C:\data\ratings.csv`. WSL exposes that directory as `/mnt/c/data`.

Run the following in a **WSL terminal**. It creates a `movielens` directory inside your HDFS user directory, then creates separate `movies` and `ratings` input directories.

```bash
test -f /mnt/c/data/movies.csv
test -f /mnt/c/data/ratings.csv

hdfs dfs -mkdir -p \
  "/user/$USER/movielens/movies" \
  "/user/$USER/movielens/ratings"

hdfs dfs -put -f \
  /mnt/c/data/movies.csv \
  "/user/$USER/movielens/movies/"

hdfs dfs -put -f \
  /mnt/c/data/ratings.csv \
  "/user/$USER/movielens/ratings/"

hdfs dfs -ls -h "/user/$USER/movielens/movies"
hdfs dfs -ls -h "/user/$USER/movielens/ratings"
```

If the CSV files are inside a subdirectory of `C:\data`, change only the two local `/mnt/c/data/...` source paths. Keep the HDFS destinations unchanged.

## 3. Connect to the Spark standalone master

Spark uses `HADOOP_CONF_DIR` to locate HDFS.

In [ ]:
import os
import socket

from pyspark.sql import SparkSession

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("D30-MovieLens-DataFrames")
    .master(master_url)
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version :", spark.version)
print("Spark master  :", sc.master)
print("Application ID:", sc.applicationId)
print("Spark UI      :", sc.uiWebUrl)

## 4. Define explicit schemas

An explicit schema avoids an extra inference pass and makes expected types clear. MovieLens identifiers are integers, ratings are doubles, and timestamps are Unix epoch seconds.

In [ ]:
from pyspark.sql.types import DoubleType, IntegerType, LongType, StringType, StructType

movie_schema = (
    StructType()
    .add("movieId", IntegerType(), nullable=False)
    .add("title", StringType(), nullable=False)
    .add("genres", StringType(), nullable=True)
)

rating_schema = (
    StructType()
    .add("userId", IntegerType(), nullable=False)
    .add("movieId", IntegerType(), nullable=False)
    .add("rating", DoubleType(), nullable=False)
    .add("timestamp", LongType(), nullable=False)
)

## 5. Read movies and ratings from HDFS

A directory path reads all compatible CSV part files in that directory.

In [ ]:
hdfs_user = os.environ["USER"]
movies_path = f"hdfs:///user/{hdfs_user}/movielens/movies"
ratings_path = f"hdfs:///user/{hdfs_user}/movielens/ratings"

movie_df = (
    spark.read
    .option("header", True)
    .schema(movie_schema)
    .csv(movies_path)
)

rating_df = (
    spark.read
    .option("header", True)
    .schema(rating_schema)
    .csv(ratings_path)
)

movie_df.printSchema()
movie_df.show(5, truncate=False)
rating_df.printSchema()
rating_df.show(5)

## 6. Validate the input

Always check counts and required keys before analysis. Null identifiers usually indicate a wrong schema, a malformed row, or an unexpected file in the input directory.

In [ ]:
from pyspark.sql.functions import col

movie_count = movie_df.count()
rating_count = rating_df.count()
invalid_movie_ids = movie_df.filter(col("movieId").isNull()).count()
invalid_rating_keys = rating_df.filter(
    col("userId").isNull() | col("movieId").isNull()
).count()

print("Movies              :", movie_count)
print("Ratings             :", rating_count)
print("Null movie IDs      :", invalid_movie_ids)
print("Null rating keys    :", invalid_rating_keys)

## 7. Explore rating values

Sorting makes the distinct scale easier to verify.

In [ ]:
rating_df.select("rating").distinct().orderBy("rating").show()

## 8. Aggregate ratings by movie

`groupBy` creates one group per `movieId`. The aggregation calculates both average rating and number of ratings. The filters retain well-rated movies with enough votes to make the average useful.

In [ ]:
from pyspark.sql.functions import avg, count, desc

popular_ratings_df = (
    rating_df
    .groupBy("movieId")
    .agg(
        avg("rating").alias("avg_rating"),
        count("userId").alias("total_ratings"),
    )
    .filter(
        (col("total_ratings") >= 100)
        & (col("avg_rating") >= 3.5)
    )
)

popular_ratings_df.orderBy(desc("total_ratings")).show(20)

## 9. Join ratings with movie titles

Joining on the shared `movieId` column keeps one copy of the key. The final ordering uses rating count first and average rating as a tie-breaker.

In [ ]:
popular_movies_df = (
    popular_ratings_df
    .join(movie_df, on="movieId", how="inner")
    .select("movieId", "title", "genres", "avg_rating", "total_ratings")
    .orderBy(desc("total_ratings"), desc("avg_rating"))
    .cache()
)

popular_movies_df.show(20, truncate=False)
print("Output partitions:", popular_movies_df.rdd.getNumPartitions())

## 10. Write distributed CSV output

Spark writes a directory, not a single CSV filename. Normally it writes one `part-*` file per output partition. This parallel behavior is preferred for large data.

In [ ]:
output_base = f"hdfs:///user/{hdfs_user}/movielens/output"
distributed_output = f"{output_base}/popular-movies"

(
    popular_movies_df.write
    .mode("overwrite")
    .option("header", True)
    .csv(distributed_output)
)

print("Saved to:", distributed_output)

## 11. Optional: write one part file

`coalesce(1)` reduces the result to one partition and therefore one data part file. Use this only for small teaching or export results; it removes parallelism and can become a bottleneck.

In [ ]:
single_part_output = f"{output_base}/popular-movies-single-part"

(
    popular_movies_df.coalesce(1).write
    .mode("overwrite")
    .option("header", True)
    .csv(single_part_output)
)

print("Saved to:", single_part_output)

## 12. Verify and read the result

In [ ]:
result_df = (
    spark.read
    .option("header", True)
    .schema(popular_movies_df.schema)
    .csv(distributed_output)
)

result_df.show(10, truncate=False)

In [ ]:
%%bash
hdfs dfs -ls -h "/user/$USER/movielens/output/popular-movies"
hdfs dfs -ls -h "/user/$USER/movielens/output/popular-movies-single-part"

## 13. Stop Spark

Unpersist the cached result and stop the session when the lesson is complete.

In [ ]:
popular_movies_df.unpersist()
spark.stop()
print("Spark session stopped.")